# PINN example notebook: Logistic Regression

This is an example notebook to learn the basics of Pysics Informed Neural Networks (PINNs) with a very simple example:|
Logistic Regression

The differential equation for logistic regression can be solved analytically. \
Using a neural network to learn this solution to the problem aims to explain the concept and application of a PINN to differential equations in general.

In [1]:
import torch 
import torchopt
from torch import nn

from functorch import make_functional
from torch.func import grad, vmap

import torch.nn as nn

NOTE: Redirects are currently not supported in Windows or MacOs.


In [66]:
class NNApproximator(nn.Module):
    def __init__(
        self,
        num_inputs: int = 1,
        num_outputs: int = 1,
        num_hidden: int = 2,
        dim_hidden: int = 2,
        act: nn.Module = nn.Tanh(),
    ) -> None:
        """Simple neural network with linear layers and non-linear activation function
        This class is used as universal function approximator for the solution of
        partial differential equations using PINNs
        Args:
            num_inputs (int, optional): The number of input dimensions
            num_outputs (int, optional): The number of outputs of the model, in general is 1
            num_hidden (int, optional): The number of hidden layers in the model
            dim_hidden (int, optional): The number of neurons for each hidden layer
            act (nn.Module, optional): The type of non-linear activation function to be used
        """
        super().__init__()

        self.layer_in = nn.Linear(num_inputs, dim_hidden)
        self.layer_out = nn.Linear(dim_hidden, num_outputs)

        num_middle = num_hidden - 1
        self.middle_layers = nn.ModuleList(
            [nn.Linear(dim_hidden, dim_hidden) for _ in range(num_middle)]
        )
        self.act = act

        self.num_inputs = num_inputs
        self.num_outputs = num_outputs

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.act(self.layer_in(x))
        for layer in self.middle_layers:
            out = self.act(layer(out))
        return self.layer_out(out)


In [67]:
# create the PINN model and make it functional using functorch utilities
model = NNApproximator()
fmodel, params = make_functional(model)

/Users/konstantinnikolaou/miniconda3/envs/pinnlab/lib/python3.10/site-packages/torch/_functorch/deprecated.py:97: UserWarning: We've integrated functorch into PyTorch. As the final step of the integration, functorch.make_functional is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use torch.func.functional_call instead; see the PyTorch 2.0 release notes and/or the torch.func migration guide for more details https://pytorch.org/docs/master/func.migrating.html
  warn_deprecated('make_functional', 'torch.func.functional_call')


In [127]:
def f(x: torch.Tensor, params: torch.Tensor) -> torch.Tensor:
    """
    Function for vmapping the model over multiple inputs. 

    Vmapping over multiple inputs results in a dimensionless tensor. 
    Therefore, in order for the model to be able to process the input,
    the input must be unsqueezed to add a batch dimension.
    For the vmapped output to be a tensor of the same shape as the input, the
    output must be squeezed again.

    Moreover, in order to compute higher order derivatives, x is passed as the
    first argument and the parameters as the second argument. 
    This is because the `grad` function of PyTorch takes the first argument as
    default for computing the gradient.

    Parameters:
    -----------
        x (torch.Tensor): The input tensor
        params (torch.Tensor): The parameters of the model
    Returns:
    --------
        res (torch.Tensor): The output of the model
    """
    # only a single element is supported thus unsqueeze must be applied
    # for batching multiple inputs, `vmap` must be used as below
    x_ = x.unsqueeze(0)
    res = fmodel(params, x_).squeeze(0)
    return res

# use `vmap` primitive to allow efficient batching of the input
f_vmap = vmap(f, in_dims=(0, None))

# return function for computing higher order gradients with respect
# to input by simply composing `grad` calls and use again `vmap` for
# efficient batching of the input

dfdx = vmap(grad(f), in_dims=(0, None))
d2fdx2 = vmap(grad(grad(f)), in_dims=(0, None))

In [136]:
R = 1.0  # rate of maximum population growth parameterizing the equation
X_BOUNDARY = 0.0  # boundary condition coordinate
F_BOUNDARY = 0.5  # boundary condition value


def loss_fn(params: torch.Tensor, x: torch.Tensor) -> torch.Tensor:

    # interior loss
    f_value = f_vmap(x, params)

    interior = dfdx(x, params) - R * f_value * (1 - f_value)

    # boundary loss
    x0 = X_BOUNDARY
    f0 = F_BOUNDARY
    x_boundary = torch.tensor([x0])
    f_boundary = torch.tensor([f0])
    boundary = f(x_boundary, params) - f_boundary

    loss = nn.MSELoss()
    loss_value = loss(interior, torch.zeros_like(interior)) + loss(
        boundary, torch.zeros_like(boundary)
    )

    return loss_value

In [138]:
# choose the configuration
batch_size = 30  # number of colocation points sampled in the domain
num_iter = 100  # maximum number of iterations
learning_rate = 1e-1  # learning rate
domain = (-5.0, 5.0)  # logistic equation domain

# choose optimizer with functional API using functorch
optimizer = torchopt.FuncOptimizer(torchopt.adam(lr=learning_rate))

# train the model
for i in range(num_iter):

    # sample colocations points in the domain randomly at each epoch
    x = torch.FloatTensor(batch_size).uniform_(domain[0], domain[1])

    # update the parameters using the functional API
    loss = loss_fn(params, x)
    params = optimizer.step(loss, params)

    print(f"Iteration {i} with loss {float(loss)}")

Iteration 0 with loss 6.935150304343551e-05
Iteration 1 with loss 0.0270782969892025
Iteration 2 with loss 0.02837609499692917
Iteration 3 with loss 0.0048439656384289265
Iteration 4 with loss 0.024351082742214203
Iteration 5 with loss 0.008441182784736156
Iteration 6 with loss 0.0030350920278578997
Iteration 7 with loss 0.014849595725536346
Iteration 8 with loss 0.01342448778450489
Iteration 9 with loss 0.0045741647481918335
Iteration 10 with loss 0.0030743153765797615
Iteration 11 with loss 0.00839147437363863
Iteration 12 with loss 0.0062789469957351685
Iteration 13 with loss 0.0019608568400144577
Iteration 14 with loss 0.004433613270521164
Iteration 15 with loss 0.005275078117847443
Iteration 16 with loss 0.0021073664538562298
Iteration 17 with loss 0.0013801086461171508
Iteration 18 with loss 0.003475262550637126
Iteration 19 with loss 0.004510519094765186
Iteration 20 with loss 0.0016514186281710863
Iteration 21 with loss 0.00015279784565791488
Iteration 22 with loss 0.0026000021

# Similar Setup with Pytorch 2.0

In [149]:
# create the PINN model using the new pytorch API
model = NNApproximator()

params = dict(model.named_parameters())
fmodel = torch.func.functionalize(model)
x = torch.rand(3, 1)
fmodel(x)

tensor([[0.5099],
        [0.5053],
        [0.5047]], grad_fn=<AddmmBackward0>)